# 10. Bronze Ingestion — Playback Heartbeats Stream
**Layer:** Medallion Architecture ➡️ **Bronze Layer** (`playback_lakehouse.bronze.playback_heartbeats`)  
**Pattern:** Incremental Streaming Ingestion via Databricks Auto Loader (`cloudFiles`)

---

## Purpose
The primary goal of this notebook is to ingest raw, high-volume JSON playback telemetry (heartbeats) from the Landing Zone managed volume into an append-only Delta Lake table. 

## Key Architectural Decisions & Patterns

### 1. Databricks Auto Loader (`format("cloudFiles")`)
We utilize Auto Loader to scale efficiently. 
* It uses optimal file discovery (bookkeeping metadata) to process only newly arrived files, eliminating the overhead of full bucket scans.
* It operates via `.trigger(availableNow=True)` providing a cost-effective "streaming pipeline that shuts down automatically" once all currently available files are processed.

### 2. Defensive Ingestion & Schema Resilience
* **Schema Evolution:** Using `.option("cloudFiles.schemaLocation", ...)` combined with `.option("mergeSchema", "true")` ensures that any new fields added downstream changes are automatically captured and integrated without manual DDL alterations or downtime.
* **Schema Hints:** We explicitly enforce types on critical driving keys (`event_ts`, `event_id`, `user_id`, `title_id`) using `schemaHints` to guarantee downstream components don't receive invalid data types.
* **Rescued Data:** Corrupted, malformed, or structural type-mismatches are caught in the hidden `_rescued_data` column, keeping the ingestion pipeline fully operational.

### 3. Lineage Enrichment
Every row is explicitly injected with metadata attributes:
* `_ingest_ts`: Wall-clock system timestamp of when Spark processed the batch.
* `_source_file`: The exact file path in the cloud object storage.

---

## Target Schema Summary
* **Input Path:** `/Volumes/playback_lakehouse/landing/raw/playback_heartbeats`
* **Output Table:** `playback_lakehouse.bronze.playback_heartbeats`
* **Format:** Delta Lake (Append-Only)

In [0]:
from conf.config import (
    SCHEMA_PATH_HEARTBEATS, RAW_HEARTBEATS, CKPT_PATH_HEARTBEATS, BRONZE_HEARTBEATS
)

In [0]:
from pyspark.sql import functions as F

bronze_stream = (
    spark.readStream
        .format("cloudFiles")  # Auto Loader
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH_HEARTBEATS)  # persisted inferred schema
        .option("cloudFiles.inferColumnTypes", "true") # infer int/long/bool, not all-strings
        .option("cloudFiles.schemaHints", # force the tricky ones
                "event_ts timestamp, event_id long, user_id long, title_id int")
        .load(RAW_HEARTBEATS)
)

# Bronze principle: keep raw as-is, but add ingestion lineage metadata
bronze_enriched = (
    bronze_stream
        .withColumn("_ingest_ts",   F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
)

query = (
    bronze_enriched.writeStream
        .format("delta")
        .option("checkpointLocation", CKPT_PATH_HEARTBEATS)
        .option("mergeSchema", "true")
        .trigger(availableNow=True) # process all available files, then STOP
        .toTable(BRONZE_HEARTBEATS)
)
query.awaitTermination()
print("Bronze ingestion complete.")